# 项目2 · 电商数据工程 ETL（Prefect + dbt + 星型模型）

重点看第 1、2 步：dbt 三层建模和数据质量测试，这是面试的核心。

In [ ]:
# ① 环境变量（每次重启内核后必须先跑这一格）
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"      # 修复 Anaconda OpenMP 冲突
os.environ["PYTHONIOENCODING"] = "utf-8"          # 中文不乱码
os.environ["HF_HUB_OFFLINE"] = "1"                # 模型走本地缓存，不联网
os.environ["TRANSFORMERS_OFFLINE"] = "1"
print("环境变量已设置")
print("如果后面报 OMP: Error #15，说明有库已经提前加载了，重启内核后再跑这一格")

In [ ]:
# ② 切到项目根目录（脚本里的相对路径都依赖这个）
import os
os.chdir(r"D:/Projects/etl_pipeline_v2")
print("当前目录:", os.getcwd())

## 1. dbt 建模（staging → intermediate → marts）

预期：`PASS=14 ERROR=0`

In [ ]:
import os
os.chdir("dbt_project")
!dbt run --profiles-dir .
os.chdir("..")

## 2. dbt 数据测试

预期：`PASS=19 ERROR=0`

In [ ]:
os.chdir("dbt_project")
!dbt test --profiles-dir .
os.chdir("..")

## 3. 查看 dbt 产出的三层模型

In [ ]:
import sqlite3

conn = sqlite3.connect("warehouse.db")
print("=== staging（视图）===")
for (n,) in conn.execute("select name from sqlite_master where type='view' and name like 'stg_%' order by name"):
    print(f"  {n:28} {conn.execute(f'select count(*) from [{n}]').fetchone()[0]:>8} 行")
print("\n=== intermediate（视图）===")
for (n,) in conn.execute("select name from sqlite_master where type='view' and name like 'int_%' order by name"):
    print(f"  {n:28} {conn.execute(f'select count(*) from [{n}]').fetchone()[0]:>8} 行")
print("\n=== marts（物化表）===")
for (n,) in conn.execute("select name from sqlite_master where type='table' and name like 'mart_%' order by name"):
    print(f"  {n:28} {conn.execute(f'select count(*) from [{n}]').fetchone()[0]:>8} 行")

## 4. 看 mart 结果长什么样（每日销售看板）

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)

df = pd.read_sql("""
    select full_date, city_name, category, order_count,
           round(total_revenue,2) as total_revenue,
           round(ma7_revenue,2)   as ma7_revenue
    from mart_sales_daily
    order by full_date desc
    limit 10
""", conn)
df

## 5. 启动 Streamlit 看板（可选）

**注意**：这一格会启动一个网页服务，需要手动停止。
想停止就点上方工具栏的 ■（中断内核）。

In [ ]:
import subprocess, webbrowser, time
proc = subprocess.Popen(["streamlit", "run", "dashboard/app.py"])
time.sleep(8)
webbrowser.open("http://localhost:8501")
print("已经在浏览器打开了，看完记得中断内核")